In [4]:
import os
import glob
import cv2
import numpy as np

def ensure_dir(path):
    """Создаёт папку, если её нет."""
    if not os.path.exists(path):
        os.makedirs(path)

def local_min_max(img_gray, kernel):
    """
    Находит локальный минимум и максимум в окрестности, 
    заданной структурирующим элементом `kernel`.
    
    Пояснение:
      - erode(img, kernel) = локальный минимум по форме kernel
      - dilate(img, kernel) = локальный максимум по форме kernel
    """
    # Применяем морфологическую эрозию/дилатацию
    f_min = cv2.erode(img_gray, kernel)
    f_max = cv2.dilate(img_gray, kernel)
    return f_min, f_max

def process_image(img_path, output_folder, 
                  shape_type="circle",  # или "rect", "point"
                  window_size=3,
                  fG=255, fH=0):
    """
    Обрабатывает изображение по формуле из задания:
      f_j^new = (1 - phi_j)*fG + phi_j*fH,
    где phi_j = (f_j - f_min) / (f_max - f_min), 
    в локальном окне (min, max) — морфологические операции.
    
    Параметры:
      - img_path: путь к исходному файлу
      - output_folder: куда сохранять результат
      - shape_type: тип ядра ("circle", "rect", "point")
      - window_size: размер структурирующего элемента (диаметр или сторона)
      - fG, fH: яркость «figure» и «background»
    """
    # Читаем в градациях серого
    img_gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        print(f"[!] Не удалось открыть: {img_path}")
        return
    
    # Создаём структурирующий элемент
    if shape_type == "circle":
        # Круглый элемент 3×3 (или window_size×window_size)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (window_size, window_size))
    elif shape_type == "rect":
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (window_size, window_size))
    elif shape_type == "point":
        # Точка (1×1); window_size игнорируем
        kernel = np.ones((1,1), np.uint8)
    else:
        # По умолчанию возьмём круг
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (window_size, window_size))
    
    # Получаем локальный min и max через морфологию
    f_min, f_max = local_min_max(img_gray, kernel)

    # Приведём к float32, чтобы безопасно делать вычисления
    img_f = img_gray.astype(np.float32)
    f_min_f = f_min.astype(np.float32)
    f_max_f = f_max.astype(np.float32)
    
    # Для каждого пикселя считаем phi_j
    # phi_j = (f_j - f_min) / (f_max - f_min), если f_max != f_min
    # иначе пусть phi_j = 0.5 (или 0, или 1 — зависит от вашего желания)
    numerator = img_f - f_min_f
    denominator = f_max_f - f_min_f
    
    # Избежим деления на 0: где denominator == 0, пусть будет 0.5
    mask_zero = (denominator == 0)
    denominator[mask_zero] = 1e-6  # временно, чтобы не делить на 0
    
    phi = numerator / denominator
    # Там, где был denominator=0, выставим phi=0.5
    phi[mask_zero] = 0.5
    
    # Теперь считаем f_j^new
    # f_j^new = (1 - phi_j)*fG + phi_j*fH
    # Если хотим наоборот, фоновый цвет 255 и символ 0 — меняем fG, fH местами
    f_new = (1.0 - phi) * fG + phi * fH
    
    # Приводим результат к uint8
    out_img = np.clip(f_new, 0, 255).astype(np.uint8)

    # Сохраняем
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    out_path = os.path.join(output_folder, f"{base_name}_{shape_type}.png")
    cv2.imwrite(out_path, out_img)
    print(f"Сохранено: {out_path}")

def main():
    # Папка с исходными изображениями
    input_folder = "source_images"
    # Папка для результатов
    output_folder = "results/task_6"
    ensure_dir(output_folder)

    # Допустим, берём все .jpeg файлы
    image_paths = glob.glob(os.path.join(input_folder, "*.png"))
    if not image_paths:
        print(f"[!] В папке {input_folder} нет файлов .png")
        return

    # Пример параметров
    # Для «изображения точки» (shape_type="point") — окно 1×1
    # Для «круга диаметром 3 пикселя» (shape_type="circle") — окно 3×3
    # Можно по заданиям менять shape_type для разных файлов/условий
    shape_types = ["point", "circle"]  # допустим, хотим протестировать оба

    for img_path in image_paths:
        # В зависимости от названия файла (или по очереди) можно выбирать:
        #   shape_type="circle"    -> круг 3×3
        #   shape_type="point"     -> точка (1×1)
        # Ниже — просто пример: обработаем 2 раза: один раз точкой, один раз кругом
        for stype in shape_types:
            if stype == "point":
                process_image(img_path, output_folder, shape_type=stype, window_size=3)
            else:
                process_image(img_path, output_folder, shape_type=stype, window_size=3)

if __name__ == "__main__":
    main()


Сохранено: results/task_6/text-c-true_point.png
Сохранено: results/task_6/text-c-true_circle.png
Сохранено: results/task_6/text-c-crc_point.png
Сохранено: results/task_6/text-c-crc_circle.png
Сохранено: results/task_6/text-c-sp_point.png
Сохранено: results/task_6/text-c-sp_circle.png


Когда вы используете **ядро 1×1** (точку) для морфологических операций, происходит следующее:

- **Локальный мин и макс** в окрестности пикселя при ядре $1\times1$ будут **равны самому этому пикселю**.  
- То есть $f_{\min,j} = f_j$ и $f_{\max,j} = f_j$.  
- Тогда разница $(f_{\max,j} - f_{\min,j}) = 0$, и в коде на этот случай $\phi_j$ ставится равным $0.5$.  
- Из формулы $f_j^\text{нов} = (1 - \phi_j)\,f_G + \phi_j\,f_H$ при $\phi_j = 0.5$ и $f_G = 255,\; f_H = 0$ мы получаем $f_j^\text{нов} = 128$.  
- В итоге **всё изображение заливается серым** (значением 128).

Поэтому при **shape_type="point"** (ядро $1\times1$) вы неизбежно получите **сплошную заливку**. Если по заданию вам действительно нужно «обработать изображение точкой» (что теоретически означает «нет локального окна»), то математически морфология сводится к тому, что локальный минимум и максимум равны пикселю. Логично, что разница будет 0, и формула даёт постоянное значение.

---

## Как исправить / что делать

1. **Если «point» в задании не нужен**  
   - Используйте ядро большего размера (3×3, 5×5 и т. д.). Тогда в каждом пикселе будет реальный разброс значений $f_{\min}\neq f_{\max}$, и результат будет нетривиальным.

2. **Если по условию «точка» обязана быть**  
   - Понять, что морфологический подход при точечном ядре **не даёт локального контекста**.  
   - Возможно, по методичке требуется показать, что результат будет именно **константным** (то есть «ничего не видно»), и это «правильное» доказательство.  
   - Если же вы хотите «сохранить исходное изображение», можно в коде прописать отдельную ветку:  
     ```python
     if shape_type == "point":
         # Просто копируем исходное изображение без изменений
         out_img = img_gray
     else:
         # ... морфологические операции ...
     ```
     Тогда «обработка точкой» фактически пропускается.

3. **Изменить логику при denominator=0**  
   - Сейчас, когда $f_{\max} = f_{\min}$, мы ставим $\phi_j = 0.5$. Можете вместо этого ставить $\phi_j = 0$ (тогда будет $f_G$) или $\phi_j = 1$ (тогда будет $f_H$), или **оставлять исходный пиксель** ($f_j^\text{нов} = f_j$). Но при ядре $1\times1$ это всё равно даст **одно и то же** во всём изображении, если не считать шум.

4. **Использовать размытие или другие операции**  
   - Если изображение «точка» — возможно, подразумевается небольшой круг (3×3) или «матричное» усреднение, а не морфологические min/max. Тогда не будет обесцвечивания.

---

### Вывод

- **Причина «серого квадрата»**: при ядре $1\times1$ нет разницы между min и max в локальном окне, отсюда $\phi_j$ становится константой.  
- **Решение**: либо не использовать ядро $1\times1$, либо специально прописать исключение в коде, если вам нужна иная логика для «point».  